# 08: Inspect grader errors

Contact sheets of the images a grader got wrong at one site, next to a sample it got right, to
see what the errors have in common (treated eyes, label noise, an unusual camera, ...). Shows the
processed 512 px crops as stored, before normalisation, captioned with p(referable) and the
original image size (a rough proxy for the camera).

The sheets contain dataset images: keep them out of the repository.

**Settings** (right-hand panel): Accelerator **None**, Internet **On**.
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → the preprocessing notebook of the site (DDR: **04**) and
  **07: Grader predictions**
- *Datasets* → *Your Datasets* → **retinaguard-manifest**

**Run:** set the parameters below, then *Save Version* → *Save & Run All (Commit)*. About 2 min.

**Output** (`/kaggle/working/inspect/`): `errors.jpg`, `correct.jpg`, `errors.csv`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
RUN_NAME = "grader-convnext_tiny"
SOURCE = "ddr"
TRUE_GRADE = 4  # errors: images of this true grade ...
PREDICTED_GRADE = 0  # ... predicted as this grade
N_CORRECT = 48  # sample of same-grade images predicted referable, for comparison

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Inputs -------------------------------------------------------------------
import re
from pathlib import Path

import pandas as pd

from retinaguard.data.processing import ProcessedImages, shard_roots

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/inspect")
OUT_DIR.mkdir(parents=True, exist_ok=True)
predictions_files = sorted(INPUT_ROOT.glob(f"**/predictions/{RUN_NAME}/predictions.parquet"))
assert predictions_files, f"predictions/{RUN_NAME} not found: attach the output of notebook 07"
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard = re.compile(rf"^{SOURCE}_\d{{3}}\.zip$")
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/*.zip") if shard.match(p.name)})
assert shard_dirs, f"no {SOURCE} shards found: attach the site's preprocessing notebook"

predictions = pd.read_parquet(predictions_files[0])
# p_grade_0 ... p_grade_4; read from the file so this notebook needs no training libraries
PROBABILITY_COLUMNS = [c for c in predictions.columns if c.startswith("p_grade_")]
rows = pd.read_parquet(manifests[0]).merge(predictions, on="image_id")
rows = rows[(rows["source"] == SOURCE) & (rows["dr_grade"] == TRUE_GRADE)].copy()
rows["predicted"] = rows[PROBABILITY_COLUMNS].to_numpy().argmax(1)
errors = rows[rows["predicted"] == PREDICTED_GRADE].sort_values("p_referable")
correct = (
    rows[rows["predicted"] >= 2]
    .sample(min(N_CORRECT, (rows["predicted"] >= 2).sum()), random_state=0)
    .sort_values("p_referable")
)
print(
    f"{SOURCE} grade {TRUE_GRADE}: {len(rows)} images, {len(errors)} predicted grade "
    f"{PREDICTED_GRADE}, showing {len(correct)} predicted referable for comparison"
)
errors[["image_id", "width", "height", "p_referable", *PROBABILITY_COLUMNS]].to_csv(
    OUT_DIR / "errors.csv", index=False
)

In [ ]:
# --- 2. Contact sheets -----------------------------------------------------------
import cv2
import numpy as np
from IPython.display import Image, display

TILE, COLUMNS = 224, 8
paths = dict(zip(rows["image_id"], rows["processed_path"], strict=True))
reader = ProcessedImages(paths, shard_roots(*shard_dirs))


def sheet(selected: pd.DataFrame, path: Path) -> None:
    tiles = []
    for row in selected.itertuples():
        tile = cv2.resize(reader.read(row.image_id), (TILE, TILE), interpolation=cv2.INTER_AREA)
        caption = f"p={row.p_referable:.2f} {row.width}x{row.height}"
        cv2.rectangle(tile, (0, TILE - 20), (TILE, TILE), (0, 0, 0), thickness=-1)
        cv2.putText(
            tile,
            caption,
            (4, TILE - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )
        tiles.append(tile)
    tiles += [np.zeros_like(tiles[0])] * (-len(tiles) % COLUMNS)
    grid = np.vstack([np.hstack(tiles[i : i + COLUMNS]) for i in range(0, len(tiles), COLUMNS)])
    cv2.imwrite(str(path), cv2.cvtColor(grid, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 88])
    display(Image(filename=str(path)))


print(f"Errors: grade {TRUE_GRADE} predicted grade {PREDICTED_GRADE} (lowest p first)")
sheet(errors, OUT_DIR / "errors.jpg")
print(f"Correct: grade {TRUE_GRADE} predicted referable (random sample)")
sheet(correct, OUT_DIR / "correct.jpg")

**After the version finishes:** download `errors.jpg` and `correct.jpg` from the **Output** tab
(keep them out of the repository) and share them.